# VALAM_AI PROJECT STRUCTURE — FULL BREAKDOWN

NOTES TO SELF SO I STOP FORGETTING THIS

RULE OF THUMB: IF IT TRAINS A MODEL OR BUILDS A SEARCHABLE DB → `scripts/`. IF IT RUNS LIVE WHEN A USER HITS THE API → `app/`.

## 1. ROOT LEVEL

```
Valam_AI/
├── app/            → THE ACTUAL BACKEND, RUNS LIVE
├── data/            → RAW DATASETS
├── scripts/         → OFFLINE JOBS I RUN MYSELF (TRAINING + INGESTING)
├── tests/           → AUTO + MANUAL TESTING
├── venv/            → LOCAL VIRTUAL ENV
├── .env             → SECRETS/CREDENTIALS
├── .gitignore       → STOP GIT FROM TRACKING JUNK (venv, .env, pycache)
└── requirements.txt → LIST OF LIBS NEEDED
```

- **venv** — ISOLATES PROJECT PACKAGES FROM SYSTEM PYTHON
- **.env** — API KEYS / DB PASSWORDS, NEVER HARDCODE THESE
- **.gitignore** — venv, .env, __pycache__ ETC SHOULDNT GO TO GITHUB
- **requirements.txt** — `pip install -r requirements.txt` READS THIS

## 2. `scripts/` — NOT JUST TRAINING, TWO JOBS LIVE HERE

```
scripts/
├── train_crop_model.py       ✅ ML — RandomForest on crop_recommendation.csv
├── train_disease_model.py    ✅ DL — CNN on PlantVillage
├── train_weedpest_model.py   ⬜ DL — CNN on DeepWeeds (BUILD THIS NEXT)
└── ingest_scheme_docs.py     ✅ NOT TRAINING — BUILDS THE RAG VECTOR DB
```

**JOB TYPE 1 — TRAINING SCRIPTS (crop, disease, weedpest)**
- LOADS DATASET → TRAINS FROM SCRATCH → SAVES MODEL FILE INTO `ml_models/`
- ONLY 3 FEATURES NEED THIS. VOICE DOES NOT (EXPLAINED BELOW)

**JOB TYPE 2 — INGEST SCRIPTS (schemes, later: pesticide, community)**
- NO MODEL TRAINING HAPPENS HERE AT ALL
- READS DOCS (PDFs/text) → CHUNKS THEM → EMBEDS THEM → SAVES INTO `vector_store/`
- THIS IS WHAT MAKES RAG WORK: AT QUERY TIME, GEMINI GETS HANDED THE RELEVANT CHUNK INSTEAD OF GUESSING
- WHY IT MATTERS: GEMINI DOESNT KNOW SPECIFIC GOVT SCHEME RULES OUT OF THE BOX. THIS SCRIPT IS WHAT TEACHES IT WHERE TO LOOK

SO scripts/ = "OFFLINE BATCH JOBS I RUN MYSELF" NOT JUST "TRAINING."

## 3. `app/` — THE LIVE BACKEND

```
app/
├── auth/            → LOGIN/JWT STUFF
├── ml_models/        → SAVED TRAINED FILES (.pt / .pkl) — OUTPUT OF scripts/train_*.py
├── models/           → DB TABLE DEFINITIONS (SQLAlchemy) — farmer.py, land.py, expense.py
├── routers/          → API ENDPOINTS, WRAPS services/
├── schemas/           → PYDANTIC — DEFINES SHAPE OF REQUEST/RESPONSE DATA
├── services/          → ALL THE ACTUAL AI LOGIC (SPLIT BELOW)
├── utils/             → HELPER FUNCS (logger.py, exceptions.py)
├── vector_store/       → WHERE ingest_scheme_docs.py ACTUALLY SAVES ITS DB
├── __init__.py
├── config.py          → READS .env, TURNS INTO USABLE SETTINGS FOR REST OF APP
├── database.py        → DB CONNECTION/SESSION SETUP
└── main.py             → CREATES FASTAPI APP + REGISTERS ALL ROUTERS
```

**GOTCHA I KEPT MIXING UP:**
- `models/` = DATABASE TABLE SHAPE (a farmer row, a land row)
- `ml_models/` = ACTUAL TRAINED AI MODEL FILES
- COMPLETELY DIFFERENT THINGS, NAMES JUST LOOK SIMILAR

**schemas vs models:**
- `schemas/` = WHAT AN API REQUEST/RESPONSE LOOKS LIKE (VALIDATION LAYER)
- `models/` = WHAT A DATABASE ROW LOOKS LIKE (STORAGE LAYER)
- THEY OFTEN LOOK SIMILAR BUT SERVE DIFFERENT JOBS

## 4. `services/` — SPLIT BY WHAT KIND OF AI IT IS

```
services/
├── ml/          → SERVES CROP MODEL (loads crop_recommender.pkl, predicts)
├── dl/          → SERVES DISEASE + WEEDPEST (loads .pt files) + VOICE (imports Whisper/TTS)
├── genai/       → ALL GEMINI/RAG FEATURES — market, pesticide, expense, weather, community, schemes, tutorials
└── external/    → 3RD PARTY DATA APIS, NO AI HERE — AGMARKNET, OpenWeatherMap, IMD
```

SERVICES = WHERE MODELS GET **USED**, NOT TRAINED.

TWO WAYS A FILE ENDS UP IN services/:
1. **SERVES A MODEL I TRAINED** — e.g `disease_model.py` loads `disease_cnn.pt` (produced by `scripts/train_disease_model.py`)
2. **IMPORTS + CALLS A MODEL SOMEONE ELSE ALREADY TRAINED** — e.g `voice_pipeline.py` does `import whisper` and just calls it, no training involved

## 5. TRAIN vs SERVE — THE PAIR PATTERN (NOT DUPLICATION)

I THOUGHT I WAS DOING SAME THING TWICE. IM NOT. HERE'S WHY:

| FILE | JOB | RUNS WHEN |
|---|---|---|
| `scripts/train_disease_model.py` | TRAINS CNN ON PLANTVILLAGE, SAVES `disease_cnn.pt` | ONCE, OFFLINE, BY ME |
| `services/dl/disease_model.py` | LOADS `disease_cnn.pt`, EXPOSES `predict(image)` | EVERY TIME A FARMER UPLOADS A PHOTO |

SAME PATTERN FOR WEEDPEST:
| FILE | JOB |
|---|---|
| `scripts/train_weedpest_model.py` | TRAINS ON DeepWeeds, SAVES `weedpest_cnn.pt` |
| `services/dl/weed_pest_model.py` | LOADS IT, EXPOSES `predict(image)` |

TRAINING PRODUCES THE FILE. SERVING LOADS THE FILE AND ANSWERS REQUESTS. NEED BOTH, NOT A DUPLICATE.

## 6. VOICE ASSISTANT — THE ONE EXCEPTION

**NO TRAINING SCRIPT FOR VOICE. EVER (UNLESS I DECIDE TO FINE-TUNE LATER).**

WHY: WHISPER (STT) AND GOOGLE TTS ARE ALREADY FULLY TRAINED BY OPENAI/GOOGLE. IM NOT TEACHING THEM ANYTHING NEW, JUST CALLING THEM LIKE A LIBRARY.

```python
import whisper
_model = whisper.load_model("small")   # NOT TRAINING — JUST LOADING PRE-BUILT WEIGHTS
```

COMPARE:

| | DISEASE DETECTION | VOICE ASSISTANT |
|---|---|---|
| MODEL EXISTS ALREADY? | NO — BUILD FROM SCRATCH | YES — WHISPER/TTS ALREADY EXIST |
| HAVE A DATASET? | YES — PLANTVILLAGE | NO, DONT NEED ONE |
| TRAINING LOOP? | YES | NO |
| WHAT I CODE | TRAINING + SERVING | ONLY SERVING (JUST CALLING) |

**WHERE IT LIVES:**
```
app/services/dl/voice_pipeline.py   → speech_to_text() + text_to_speech()
app/routers/voice.py                → API ENDPOINT, ORCHESTRATES THE FLOW
```

**FLOW:**
```
FARMER SPEAKS
   → routers/voice.py RECEIVES AUDIO
   → voice_pipeline.speech_to_text() (WHISPER) → TEXT
   → TEXT GETS HANDED TO WHICHEVER SERVICE NORMALLY HANDLES IT (genai/ml/dl)
   → GET BACK REPLY TEXT
   → voice_pipeline.text_to_speech() (TTS) → AUDIO
   → SENT BACK TO FARMER
```

VOICE IS A WRAPPER AROUND EXISTING LOGIC, NOT A NEW FEATURE THAT DUPLICATES ANYTHING.

**ONLY FUTURE EXCEPTION:** IF WHISPER STRUGGLES WITH RURAL TAMIL ACCENTS / FARM VOCAB, COULD FINE-TUNE LATER USING AI4BHARAT SPEECH CORPORA. NOT PART OF V1. NOT DOING IT NOW.

## 7. QUICK REFERENCE — WHICH FEATURE GOES WHERE

| FEATURE | TRAINS A MODEL? | scripts/ FILE | services/ FILE |
|---|---|---|---|
| CROP SUGGESTION | YES (ML) | train_crop_model.py | ml/crop_predictor.py |
| DISEASE DETECTION | YES (DL) | train_disease_model.py | dl/disease_model.py |
| WEED/PEST DETECTION | YES (DL) | train_weedpest_model.py | dl/weed_pest_model.py |
| VOICE ASSISTANT | NO (PRE-TRAINED) | — NONE — | dl/voice_pipeline.py |
| MARKET PRICE ADVISOR | NO (PROMPT) | — NONE — | genai/market.py |
| PESTICIDE ADVISOR | NO (RAG) | ingest_pesticide_docs.py (FUTURE) | genai/pesticide.py |
| EXPENSE TRACKER | NO (EXTRACTION) | — NONE — | genai/expense.py |
| WEATHER SPRAY ADVISORY | NO (PROMPT) | — NONE — | genai/weather.py |
| COMMUNITY ASSISTANT | NO (RAG) | ingest_community_posts.py (FUTURE) | genai/community.py |
| GOVT SCHEME ASSISTANT | NO (RAG) | ingest_scheme_docs.py ✅ | genai/schemes.py |
| TUTORIAL LIBRARY ASSISTANT | NO (SUMMARIZE/TRANSLATE) | — NONE — | genai/tutorials.py |

## 8. TL;DR — THE ONE RULE THAT COVERS EVERYTHING

```
scripts/  = OFFLINE, RUN BY ME, BUILDS SOMETHING (A MODEL FILE OR A VECTOR DB)
app/      = LIVE, RUNS ON EVERY USER REQUEST, USES WHAT scripts/ BUILT (OR IMPORTS SOMEONE ELSE'S)
```

NEXT TO BUILD: `scripts/train_weedpest_model.py` (CURRENTLY EMPTY).